# 04 — Mo Hinh Hoa: Regression + F1 Bins
Baseline - Linear - Ridge - RF - XGBoost
Metrics: MAE, RMSE, R2, F1-macro

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data.loader         import load_config
from src.features.builder    import FeatureBuilder
from src.models.supervised   import Trainer
from src.evaluation.metrics  import (regression_metrics,
                                      bin_classification_metrics,
                                      rare_zone_analysis)
from src.evaluation.report   import Reporter
from src.visualization.plots import (plot_model_comparison,
                                      plot_actual_vs_pred,
                                      plot_rare_zone)

cfg      = load_config("../configs/params.yaml")
df_clean = pd.read_parquet("../data/processed/yield_clean.parquet")
reporter = Reporter(cfg)
print("Setup OK")

## 4.1 Chuan bi du lieu

In [ ]:
fb = FeatureBuilder(cfg)
fb.fit_transform(df_clean)
print(f"Train: {len(fb.X_train):,}  Test: {len(fb.X_test):,}")

## 4.2 Huan luyen tat ca mo hinh

In [ ]:
trainer = Trainer(cfg)
trainer.fit_all(fb.X_train, fb.X_test, fb.y_train, fb.y_test,
                fb.X_train_s, fb.X_test_s)

reporter.print_model_comparison(trainer.df_results_)
reporter.save_table(trainer.df_results_, "model_comparison.csv")

In [ ]:
plot_model_comparison(trainer.df_results_, cfg["paths"]["figures"])
plt.show()

## 4.3 Phan tich loi mo hinh tot nhat

In [ ]:
best_name   = trainer.best_model_name()
y_pred_best = trainer.preds_[best_name]
print(f"Mo hinh tot nhat: {best_name}")

plot_actual_vs_pred(fb.y_test, y_pred_best, best_name, cfg["paths"]["figures"])
plt.show()

## 4.4 Feature Importance

In [ ]:
fi = trainer.feature_importance(cfg["models"]["features"])
if not fi.empty:
    fig, ax = plt.subplots(figsize=(9,4))
    fi.plot(kind="bar", color="steelblue", edgecolor="white", alpha=0.85, ax=ax)
    ax.set_title("Random Forest --- Feature Importance", fontweight="bold")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    plt.tight_layout(); plt.show()
    print(fi)

## 4.5 F1 Bin Classification + Phan tich loi vung hiem

In [ ]:
y_pred_rf = trainer.preds_.get("RandomForest", y_pred_best)

bin_result = bin_classification_metrics(fb.y_test, y_pred_rf, cfg)
print(f"F1-macro (bins Low/Medium/High): {bin_result['F1_macro']}")
print("\nF1 per class:", bin_result["F1_per_class"])
print("\nClassification Report:")
print(bin_result["classification_report"])
print("\nConfusion Matrix:")
print(bin_result["confusion_matrix"])

In [ ]:
df_zone = rare_zone_analysis(fb.y_test.values, y_pred_rf)
reporter.print_rare_zone(df_zone)
reporter.save_table(df_zone, "rare_zone_analysis.csv")

plot_rare_zone(df_zone, cfg["paths"]["figures"])
plt.show()